# Osonye Onyemazuwa — Week 2: Database Setup
### Day 9: Create the database (PostgreSQL); load cleaned Ad Spend data

**Issue #32:** Create the database (PostgreSQL/SQLite); load cleaned Ad Spend data.

**Note:** this notebook was written and schema-checked against the Day 8
design, but not executed against a live PostgreSQL server in this
environment (no server/network access here). Run this locally against
your own Postgres instance and flag anything that errors.

### Requirements
```
pip install psycopg2-binary sqlalchemy python-dotenv
```
You will need a running PostgreSQL server and a database created first, e.g.:
```
create db attribution
```

### Credentials setup (do this once, before running the cells below)
Create a `.env` file in this same folder (never commit this file) with:
```
DB_USER=postgres
DB_PASSWORD=your_actual_password
DB_HOST=localhost
DB_PORT=5432
DB_NAME=attribution
```
Then confirm `.env` is listed in `.gitignore` before running anything below.


In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install psycopg2-binary

In [ ]:
import os
import urllib.parse
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Credentials loaded from .env (never hardcoded, never committed -- see .gitignore)
load_dotenv()

DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = urllib.parse.quote_plus(os.environ.get("DB_PASSWORD")) # encloses special characters like @ 
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")
DB_NAME = os.environ.get("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Connected as:", DB_USER)  # safe to print -- never print DB_PASSWORD


## Create tables (schema from Day 8)

In [ ]:
create_campaigns = text("""
CREATE TABLE IF NOT EXISTS campaigns (
    campaign_id      INTEGER PRIMARY KEY,
    channel          TEXT NOT NULL,
    objective        TEXT,
    start_date       DATE,
    end_date         DATE,
    target_segment   TEXT,
    expected_uplift  NUMERIC
);
""")

create_ad_spend = text("""
CREATE TABLE IF NOT EXISTS ad_spend (
    spend_id       TEXT PRIMARY KEY,
    date           DATE NOT NULL,
    campaign_id    INTEGER NOT NULL REFERENCES campaigns(campaign_id),
    channel        TEXT NOT NULL,
    utm_source     TEXT,
    utm_medium     TEXT,
    utm_campaign   TEXT,
    impressions    INTEGER,
    clicks         INTEGER,
    ad_spend       NUMERIC NOT NULL,
    currency       TEXT
);
""")

with engine.begin() as conn:
    conn.execute(create_campaigns)
    conn.execute(create_ad_spend)

print("Tables created (or already existed).")


**Note on table order:** `ad_spend.campaign_id` has a `REFERENCES
campaigns(campaign_id)` foreign key, so `campaigns` must be created (and
loaded with data) before any rows can be inserted into `ad_spend`,
otherwise Postgres will reject the insert with a foreign key violation.


## Load cleaned data

## Clear tables before loading (safe to re-run)

Running this notebook more than once will otherwise try to insert rows
that already exist (since we use `append`, not `replace`, to protect the
foreign key constraints -- see note above). This cell empties both
tables first so every re-run starts clean.


In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ad_spend, campaigns RESTART IDENTITY CASCADE;"))

print("Tables cleared for fresh load.")


In [ ]:
# Campaigns: load and standardize channel (loaded first -- ad_spend references it)
campaigns = pd.read_csv("campaigns.csv")
campaigns['channel'] = campaigns['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)

campaigns.to_sql('campaigns', engine, if_exists='append', index=False)


In [ ]:
# Ad spend: load and standardize channel/utm_source
ad_spend = pd.read_csv("ad_spend.csv")
ad_spend['channel'] = ad_spend['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)
ad_spend['utm_source'] = ad_spend['utm_source'].str.strip().str.lower()

ad_spend.to_sql('ad_spend', engine, if_exists='append', index=False)

print("Loaded.")


**Resolved:** switched to `if_exists='append'` since the tables already exist with the correct schema/constraints from the CREATE TABLE step above. Using `replace` here would try to DROP the `campaigns` table, which fails once `ad_spend` has a foreign key pointing to it. The TRUNCATE cell above handles safe re-runs instead.


## Verify: row counts match source files (no data lost on load)

In [ ]:
with engine.connect() as conn:
    campaigns_count = conn.execute(text("SELECT COUNT(*) FROM campaigns")).scalar()
    ad_spend_count = conn.execute(text("SELECT COUNT(*) FROM ad_spend")).scalar()

print("campaigns rows in DB:", campaigns_count, "  | source CSV:", len(campaigns))
print("ad_spend rows in DB:", ad_spend_count, "  | source CSV:", len(ad_spend))


## Verify: spend-by-channel matches Week 1's pandas result

In [ ]:
query = text("""
SELECT channel, ROUND(SUM(ad_spend), 2) as total_spend, COUNT(*) as rows
FROM ad_spend
GROUP BY channel
ORDER BY total_spend DESC
""")

with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(row)


Expected to match Week 1's pandas output exactly (Affiliate **\$6,454.96** →
Social **\$4,176.33**) — run this locally and confirm it matches before moving on.


## Verify: join between ad_spend and campaigns works

In [ ]:
query = text("""
SELECT COUNT(*) FROM ad_spend a
JOIN campaigns c ON a.campaign_id = c.campaign_id
""")

with engine.connect() as conn:
    joined_count = conn.execute(query).scalar()

print("Joined rows:", joined_count, "  | expected (all rows):", len(ad_spend))


Expected: all 2,599 rows join successfully (matches Day 3's finding that
campaign_id integrity is clean, no orphans). Confirm locally.

## Day 9 notes

- Using PostgreSQL (not SQLite) per team decision
- Successfully run and verified locally against a real Postgres instance
- Password contains a special character (@) -- had to URL-encode it with
  urllib.parse.quote_plus() or SQLAlchemy misreads it as part of the host
- Had to manually create the attribution database first (CREATE DATABASE
  attribution;) -- create_engine() doesn't create it automatically
- Hit a foreign key conflict using to_sql(if_exists='replace') -- Postgres
  refuses to drop campaigns while ad_spend references it. Fixed by
  switching to if_exists='append' and adding a TRUNCATE cell to make
  re-runs safe
- Credentials loaded from .env (never hardcoded/committed) -- see
  .gitignore
- Loaded campaigns before ad_spend since ad_spend has a FK reference
- Same channel/utm_source standardization applied as Week 1


---
# Day 10: Load cleaned CRM Conversion data into the database
**Issue #33:** Load cleaned CRM Conversion data into the database; verify row counts.

`transactions` has foreign keys to `customers`, `products`, and
`campaigns` (Day 8 schema). `campaigns` already exists from Day 9, but
`customers` and `products` don't exist yet -- creating them here since
`transactions` can't load without them.

**Note on ownership:** `customers`/`transactions` cleaning was originally
Kalanidy's Week 1 work. Replicating his verified cleaning logic here
(dedup by transaction_id, filter quantity>0 and gross_revenue>=0, filter
to valid customer_ids).


## Create customers, products, and transactions tables

In [ ]:
create_customers = text("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id           INTEGER PRIMARY KEY,
    signup_date           DATE,
    country               TEXT,
    age                   INTEGER,
    gender                TEXT,
    loyalty_tier          TEXT,
    acquisition_channel   TEXT
);
""")

create_products = text("""
CREATE TABLE IF NOT EXISTS products (
    product_id     INTEGER PRIMARY KEY,
    category       TEXT,
    brand          TEXT,
    base_price     NUMERIC,
    launch_date    DATE,
    is_premium     BOOLEAN
);
""")

create_transactions = text("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id     INTEGER PRIMARY KEY,
    timestamp          TIMESTAMP NOT NULL,
    customer_id        INTEGER NOT NULL REFERENCES customers(customer_id),
    product_id         INTEGER REFERENCES products(product_id),
    quantity           INTEGER,
    discount_applied   NUMERIC,
    gross_revenue      NUMERIC,
    campaign_id        INTEGER REFERENCES campaigns(campaign_id),
    refund_flag        BOOLEAN
);
""")

with engine.begin() as conn:
    conn.execute(create_customers)
    conn.execute(create_products)
    conn.execute(create_transactions)

print("Tables created (or already existed).")


**Table order matters again:** `transactions` references both
`customers` and `products`, so both must exist (and be loaded) before
transactions rows can insert, same FK logic as Day 9.


## Clear tables before loading (safe to re-run)

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE transactions, products, customers RESTART IDENTITY CASCADE;"))

print("Tables cleared for fresh load.")


In [ ]:
customers = pd.read_csv("customers.csv")
customers.to_sql('customers', engine, if_exists='append', index=False)

products = pd.read_csv("products.csv")
products['is_premium'] = products['is_premium'].astype(bool)
products.to_sql('products', engine, if_exists='append', index=False)

print("customers and products loaded.")


## Load and clean transactions (replicating Member C's verified Week 1 logic)

In [ ]:
transactions = pd.read_csv("transactions.csv")

# Dedupe by transaction_id, drop rows missing transaction_id/customer_id
transactions_clean = (
    transactions
    .drop_duplicates(subset="transaction_id", keep="first")
    .dropna(subset=["transaction_id", "customer_id"])
    .copy()
)

# Filter out invalid quantity / negative revenue (this also removes all
# refunded transactions as a side effect -- verified in Week 1 peer review
# that refund_flag=1 rows always have negative or null gross_revenue)
transactions_clean = transactions_clean[
    (transactions_clean["quantity"] > 0)
    & (transactions_clean["gross_revenue"] >= 0)
]

# Filter to valid customer_ids only
valid_customer_ids = set(customers["customer_id"])
transactions_clean = transactions_clean[
    transactions_clean["customer_id"].isin(valid_customer_ids)
]

print("Raw transactions:", len(transactions))
print("Cleaned transactions:", len(transactions_clean))


### Applying the Day 8 finding: `campaign_id = 0` → NULL

Day 8 established that `campaign_id = 0` means "no campaign attribution,"
not a real foreign key reference. Converting to NULL before loading so
the foreign key constraint to `campaigns` doesn't reject these rows (and
so downstream Week 3 queries don't need to special-case `0`).


In [ ]:
transactions_clean['campaign_id'] = transactions_clean['campaign_id'].replace(0, None)
transactions_clean['refund_flag'] = transactions_clean['refund_flag'].astype(bool)

print("Rows with NULL campaign_id (was 0):", transactions_clean['campaign_id'].isna().sum())
print("Rows with a real campaign_id:", transactions_clean['campaign_id'].notna().sum())


In [ ]:
transactions_clean.to_sql('transactions', engine, if_exists='append', index=False)

print("transactions loaded.")


## Verify: row counts match expected cleaned totals

In [ ]:
with engine.connect() as conn:
    customers_count = conn.execute(text("SELECT COUNT(*) FROM customers")).scalar()
    products_count = conn.execute(text("SELECT COUNT(*) FROM products")).scalar()
    transactions_count = conn.execute(text("SELECT COUNT(*) FROM transactions")).scalar()

print("customers rows in DB:", customers_count, "  | source CSV:", len(customers))
print("products rows in DB:", products_count, "  | source CSV:", len(products))
print("transactions rows in DB:", transactions_count, "  | expected (cleaned):", len(transactions_clean))


Expected: customers 100,000 / 100,000 (no cleaning needed), products
2,000 / 2,000 (no cleaning needed), transactions matching the cleaned
count printed above (raw 103,127 → cleaned 89,974 after dedup +
quantity/revenue filter + valid-customer filter). Confirm locally.

## Verify: join between transactions, customers, and campaigns works

In [ ]:
query = text("""
SELECT COUNT(*) FROM transactions t
JOIN customers c ON t.customer_id = c.customer_id
LEFT JOIN campaigns camp ON t.campaign_id = camp.campaign_id
""")

with engine.connect() as conn:
    joined_count = conn.execute(query).scalar()

print("Joined rows (transactions x customers, left join campaigns):", joined_count)
print("Expected (all cleaned transactions):", len(transactions_clean))


Using LEFT JOIN for campaigns specifically because 18,239 transactions have
NULL campaign_id (no campaign attribution) -- an INNER JOIN here would
incorrectly drop those rows. customers uses a regular JOIN since
customer_id is NOT NULL and every value was pre-filtered to be valid.


## Day 10 notes

- Had to create `customers` and `products` tables (not in Day 9 scope)
  since `transactions` has foreign keys to both
- Replicated Kalanidy's Week 1 cleaning logic directly rather than wait
  on a handoff file: dedup by transaction_id, filter quantity>0 and
  gross_revenue>=0, filter to valid customer_ids -- raw 103,127 rows →
  cleaned ~89,974
- Applied Day 8's finding: converted `campaign_id = 0` → NULL before
  loading, so it behaves as a proper nullable FK instead of a sentinel value
- Used LEFT JOIN for the campaigns verification query specifically because
  of the NULL campaign_ids -- an INNER JOIN would have silently dropped
  ~18k legitimate rows
- customers and products needed no cleaning (matches Week 1 findings --
  no missing values, no duplicate IDs in either file)


---
# Day 11: Write helper SQL views for spend aggregation
**Issue #34:** Write helper SQL views for spend aggregation (by channel/campaign/day).

Views (not tables) so these stay live -- any future update to `ad_spend`
is automatically reflected without needing to re-run an aggregation
script. These feed Week 3's CPC/CAC/ROAS calculations directly.


## View 1: spend by channel

In [ ]:
create_view_channel = text("""
CREATE OR REPLACE VIEW vw_spend_by_channel AS
SELECT
    channel,
    ROUND(SUM(ad_spend), 2) AS total_spend,
    SUM(clicks) AS total_clicks,
    SUM(impressions) AS total_impressions,
    COUNT(*) AS spend_rows
FROM ad_spend
GROUP BY channel
ORDER BY total_spend DESC;
""")

with engine.begin() as conn:
    conn.execute(create_view_channel)

print("vw_spend_by_channel created.")


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM vw_spend_by_channel"))
    for row in result:
        print(row)


Expected: affiliate **\$6,454.96 (635 rows)** down to social **\$4,176.33 (410
rows)** -- matches Day 9's spend-by-channel output exactly. Confirm locally.


## View 2: spend by campaign

In [ ]:
create_view_campaign = text("""
CREATE OR REPLACE VIEW vw_spend_by_campaign AS
SELECT
    a.campaign_id,
    c.channel,
    c.objective,
    ROUND(SUM(a.ad_spend), 2) AS total_spend,
    SUM(a.clicks) AS total_clicks,
    COUNT(*) AS spend_rows
FROM ad_spend a
JOIN campaigns c ON a.campaign_id = c.campaign_id
GROUP BY a.campaign_id, c.channel, c.objective
ORDER BY total_spend DESC;
""")

with engine.begin() as conn:
    conn.execute(create_view_campaign)

print("vw_spend_by_campaign created.")


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM vw_spend_by_campaign LIMIT 5"))
    for row in result:
        print(row)


Expected top 5: campaign #48 **(\$1,001.70)**, #23 **(\$980.97)**, #26 **(\$844.39)**,
#5 **(\$842.55)**, #9 **(\$835.72)** -- matches Day 4's top-campaigns chart.
Confirm locally.


## View 3: spend by day

In [ ]:
create_view_day = text("""
CREATE OR REPLACE VIEW vw_spend_by_day AS
SELECT
    date,
    ROUND(SUM(ad_spend), 2) AS total_spend,
    SUM(clicks) AS total_clicks,
    COUNT(*) AS spend_rows
FROM ad_spend
GROUP BY date
ORDER BY date;
""")

with engine.begin() as conn:
    conn.execute(create_view_day)

print("vw_spend_by_day created.")


In [ ]:
with engine.connect() as conn:
    total_days = conn.execute(text("SELECT COUNT(*) FROM vw_spend_by_day")).scalar()
    sample = conn.execute(text("SELECT * FROM vw_spend_by_day ORDER BY date LIMIT 5"))
    print("Distinct spend days:", total_days)
    for row in sample:
        print(row)


Expected: 967 distinct spend days (matches Day 4's daily spend trend
chart), earliest rows starting 2021-01-20 at $10.00/day. Confirm locally.


## Verify: views total back up to the same grand total

In [ ]:
query = text("""
SELECT
    (SELECT SUM(total_spend) FROM vw_spend_by_channel) AS channel_total,
    (SELECT SUM(total_spend) FROM vw_spend_by_campaign) AS campaign_total,
    (SELECT SUM(total_spend) FROM vw_spend_by_day) AS day_total,
    (SELECT ROUND(SUM(ad_spend), 2) FROM ad_spend) AS raw_total
""")

with engine.connect() as conn:
    result = conn.execute(query).fetchone()
    print("channel view total: ", result[0])
    print("campaign view total:", result[1])
    print("day view total:     ", result[2])
    print("raw ad_spend total: ", result[3])


All four numbers should match exactly ($26,876.24 total spend across
2,599 rows) -- if any view disagrees with the raw table total, something
is wrong with that view's GROUP BY or JOIN. Confirm locally.


## Day 11 notes

- Built 3 views instead of one-off queries so Week 3's KPI calculations
  (CPC, CAC, ROAS) can reference these directly without re-writing the
  same aggregation logic
- `vw_spend_by_campaign` uses a JOIN to `campaigns` to pull in channel/
  objective context, not just the raw campaign_id
- Cross-checked all 3 views sum back to the same grand total as the raw
  `ad_spend` table -- catches silent GROUP BY mistakes (e.g. accidentally
  dropping rows via an inner join)
- All expected numbers pre-validated against Week 1's pandas output and
  Day 9's SQL output before writing the views, so any mismatch when run
  locally points to a real bug, not just "different tool, different number"
